In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad

#### LOAD IN DATA ####
This is the 28 patterns using all genes

In [ ]:
#plotDir = '/mnt/morbo/Data/Users/kwoyshner/cerebellum/results/humans_dropOutlier_reanalysis/plots_allGenes'
geneWeights = pd.read_csv('/mnt/morbo/Data/Users/kwoyshner/cerebellum/FORGITHUB/cogaps/human_CB_cogaps_n30_nIterations15k_allGenes_geneWeights.csv', index_col=0)
patterns = pd.read_csv('/mnt/morbo/Data/Users/kwoyshner/cerebellum/FORGITHUB/cogaps/human_CB_cogaps_n30_nIterations15k_allGenes_patterns.csv',index_col=0).T
print(geneWeights.shape)
print(patterns.shape)

In [ ]:
coords = pd.read_csv('/mnt/morbo/Data/Users/kwoyshner/cerebellum/data/coords_human.csv',index_col=0)
meta = pd.read_csv('/mnt/morbo/Data/Users/kwoyshner/cerebellum/data/human_CB_meta.csv', index_col=0)
#meta['Patient_Replicate'] = meta.apply(lambda x: str(x['patient']) + '_' + x['replicate'], axis=1)

clinical_meta = pd.read_excel('/mnt/morbo/Data/Users/kwoyshner/cerebellum/FORGITHUB/cogaps/SpatialSeq subject Info(Updated)_subjectAdjusted_newIDs.xlsx')
clinical_meta['Subject_newID'] = clinical_meta['Subject_newID'].fillna('NA')

clinical_meta_cell = meta.reset_index().merge(clinical_meta, left_on='patient', right_on='Subject', how='left').set_index('index')
clinical_meta_cell['Patient_Replicate'] = clinical_meta_cell.apply(lambda x: str(x['Subject_newID']) + '_' + x['replicate'], axis=1)
clinical_meta_cell['PatientGroup_title'] = clinical_meta_cell.apply(lambda x: str(x['Subject_newID']) + '_' + x['replicate'] + ' (GA ' + str(x['Gestational Age (Weeks)']) +' wks) ', axis=1)

n_patterns = patterns.shape[1]
patterns.columns = ['H-' + x.split('_')[-1] + '     ' for x in patterns.columns] # extra spaces to force the horizontal alignment of y tick labels
print(n_patterns)
patterns.head()

##### NOTE PT IS INCORRECT ####

In [ ]:
sorted_meta = clinical_meta_cell.loc[patterns.index,:].reset_index().groupby('Patient_Replicate').agg('first').sort_values(['Gestational Age (Weeks)','Patient_Replicate'])
#sorted_meta.iloc[:,50:60]
images = [x.split('_')[0] for x in sorted_meta.loc[:,'index']] # order these by gestational age
print(len(images))

#### Plot all patterns ####

In [ ]:
### PATTERNS
for pat in patterns.columns:
    fig,axes = plt.subplots(6,4,figsize=(20,20),sharex=True,sharey=True)
    fig.suptitle(pat, y=0.96, fontsize=16)

    i = 0
# pmax = patterns[pat].max()
    for c in range(axes.shape[1]):
        for r in range(axes.shape[0]):
            if i < len(images):
                im = images[i]
                patterns_im = patterns[patterns.index.str.contains(im)]
                axes[r,c].set(aspect='equal')
                scat = axes[r,c].scatter(coords.loc[patterns_im.index,'imagerow'],coords.loc[patterns_im.index,'imagecol'], s=3, alpha=0.9,c=patterns_im.loc[:,pat],cmap='viridis',vmax=1, vmin=0)
                sample = clinical_meta_cell.loc[patterns_im.index,'PatientGroup_title']
                axes[r,c].set_yticklabels([])
                axes[r,c].set_xticklabels([])
                axes[r,c].set_xticks([])
                axes[r,c].set_yticks([])
                axes[r,c].set_title(sample.unique()[0])
                fig.colorbar(scat, ax=axes[r,c])
            else:
                axes[r,c].axis('off')
            i+=1
    plt.subplots_adjust(wspace=0.05, hspace=0.3)
    fig.tight_layout(rect=[0, 0, 1, 0.95])


    #plt.savefig(plotDir + '/' + str(pat) + '.png')



### Check patient specific ###

In [ ]:
sns.clustermap(patterns.merge(pd.get_dummies(clinical_meta_cell.loc[:,'Patient_Replicate']), left_index=True, right_index=True).corr('spearman').loc[patterns.columns, clinical_meta_cell.loc[:,'Patient_Replicate'].unique()], cmap='bwr',center=0)
plt.title('Patterns Correlation with Patient Replicate, spearman')

sns.clustermap(patterns.merge(pd.get_dummies(clinical_meta_cell.loc[:,'Patient_Replicate']), left_index=True, right_index=True).corr('pearson').loc[patterns.columns, clinical_meta_cell.loc[:,'Patient_Replicate'].unique()], cmap='bwr',center=0)
plt.title('Patterns Correlation with Patient Replicate, pearson')


In [ ]:
sns.clustermap(patterns.merge(pd.get_dummies(clinical_meta_cell.loc[:,'Subject_newID']), left_index=True, right_index=True).corr('spearman').loc[patterns.columns, clinical_meta_cell.loc[:,'Subject_newID'].unique()], cmap='bwr',center=0, annot=True)
plt.title('Patterns Correlation with Patient, spearman')

sns.clustermap(patterns.merge(pd.get_dummies(clinical_meta_cell.loc[:,'Subject_newID']), left_index=True, right_index=True).corr('pearson').loc[patterns.columns, clinical_meta_cell.loc[:,'Subject_newID'].unique()], cmap='bwr',center=0, annot=True)
plt.title('Patterns Correlation with Patient, pearson')


### Drop patient specific ###

In [ ]:
### THIS IS BASED ON PEARSON CORRELATION ###
# drop 1, 3, 4, 5, 10, 13, 15, 20, 23, 25
patterns_cleaned = patterns.drop(columns=['H-1     ','H-3     ','H-4     ','H-5     ','H-10     ','H-13     ','H-15     ','H-20     ','H-23     ','H-25     ']) # observed to be patient specific
n_patterns_cleaned = patterns_cleaned.shape[1]

### Drop outlier patient from analysis ###


In [ ]:
print(clinical_meta_cell.shape)
clinical_meta_cell_sub = clinical_meta_cell[clinical_meta_cell.loc[:,'Subject_newID'] != 'NA']
print(clinical_meta_cell_sub.shape)
patterns_cleaned_sub = patterns_cleaned.loc[clinical_meta_cell_sub.index,:]
print(patterns_cleaned_sub.shape)

### Correlate with metadata ###

In [ ]:
# Categories: assigned_cell_type, layer, Cell_Type, treatment, granular_layer, GROUP
# Continuous: AGEDAYSTOTAL, AGEDAYS, Gestational Age (Weeks)

In [ ]:
def summarize_metadata(df, cat):
    """
    Summarizes the number of spots, patients, and patients with >= 3 and >= 10 spots for a given metadata category.
    """
    # Ensure the category is in the clinical metadata
    if cat not in clinical_meta_cell.columns:
        print(f"Category '{cat}' not found in clinical metadata.")
        return
    spots_per_cat = df[cat].value_counts().sort_index()     # Total spots per category
    patients_per_cat = df.groupby(cat)['Subject_newID'].nunique().sort_index()     # Unique patients per category
    spots_per_patient = df.groupby([cat, 'Subject_newID']).size().reset_index(name='n_spots')     # Spots per patient per category
    patients_3plus = spots_per_patient[spots_per_patient['n_spots'] >= 3].groupby(cat)['Subject_newID'].nunique()     # Patients with >= 3 spots per category
    patients_10plus = spots_per_patient[spots_per_patient['n_spots'] >= 10].groupby(cat)['Subject_newID'].nunique()     # Patients with >= 10 spots per category

    # Combine into a single summary DataFrame
    summary = pd.DataFrame({
        'n_spots': spots_per_cat,
        'n_patients': patients_per_cat,
        'n_patients_≥3_spots': patients_3plus,
        'n_patients_≥10_spots': patients_10plus
    }).fillna(0).astype(int)  # Fill NAs with 0 and convert to int
    
    #print(f"\nSummary for metadata: {cat}")
    #print(summary)
    display(summary)



In [ ]:
### Preview metadata 
clinical_meta_cell_sub['Decreased Gestational Age']  = clinical_meta_cell_sub.loc[:,'Gestational Age (Weeks)'] * -1
clinical_meta_cell_sub['Decreased Postmenstrual Age at Death']  = clinical_meta_cell_sub.loc[:,'Age at Death (weeks)'] * -1

cont_cats = ['AGEDAYSTOTAL', 'Age at Death (weeks)', 'Gestational Age (Weeks)']
cont_cats2 = ['Decreased Gestational Age','Decreased Postmenstrual Age at Death']

cats = ['assigned_cell_type', 'layer', 'Cell_Type', 'granular_layer', 'GROUP'] 


In [ ]:
### PLOT THE META DATA
### NOTE THE CATEGORIES THAT HAVE TOO FEW VALUES

sorted_meta_sub = clinical_meta_cell_sub.reset_index().groupby('Patient_Replicate').agg('first').sort_values(['Gestational Age (Weeks)','Patient_Replicate'])
images_sub = [x.split('_')[0] for x in sorted_meta_sub.loc[:,'index']] # order these by gestational age

for cat in cats:
    fig,axes = plt.subplots(6,4,figsize=(20,20),sharex=True,sharey=True)
    fig.suptitle(cat, y=0.96, fontsize=16)

    # Map categories to colors
    categories = clinical_meta_cell_sub.loc[:, cat].unique()
    color_map = dict(zip(categories, plt.cm.tab10.colors[:len(categories)]))  # or use sns.color_palette()

    i = 0
# pmax = patterns[pat].max()
    for c in range(axes.shape[1]):
        for r in range(axes.shape[0]):
            if i < len(images_sub):
                im = images_sub[i]
                patterns_im = patterns_cleaned_sub[patterns_cleaned_sub.index.str.contains(im)]
                axes[r,c].set(aspect='equal')

                # Get categorical variable
                sample = clinical_meta_cell.loc[patterns_im.index, cat]
                
                # Convert to colors
                cat_colors = sample.map(color_map)

                # Plot using mapped colors
                scat = axes[r,c].scatter(
                    coords.loc[patterns_im.index,'imagerow'],
                    coords.loc[patterns_im.index,'imagecol'],
                    s=3,
                    alpha=0.9,
                    c=cat_colors
                )

                sample = clinical_meta_cell.loc[patterns_im.index,'PatientGroup_title']
                axes[r,c].set_yticklabels([])
                axes[r,c].set_xticklabels([])
                axes[r,c].set_xticks([])
                axes[r,c].set_yticks([])
                axes[r,c].set_title(sample.unique()[0])
                #fig.colorbar(scat, ax=axes[r,c])
            else:
                axes[r,c].axis('off')
            i+=1
    plt.subplots_adjust(wspace=0.05, hspace=0.3)
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    # Add a single legend to the figure (outside all subplots)
    handles = [plt.Line2D([0], [0], marker='o', color='w', label=cat, markerfacecolor=color_map[cat], markersize=6) for cat in categories]
    fig.legend(handles=handles, loc='upper right', bbox_to_anchor=(1.1, 0.25), fontsize=18)

    #plt.savefig(plotDir + '/' + str(pat) + '.png', dpi=300)

    summarize_metadata(clinical_meta_cell_sub, cat)

In [ ]:
corr = patterns_cleaned_sub.merge(clinical_meta_cell.loc[:,cont_cats], left_index=True, right_index=True).corr().iloc[:n_patterns_cleaned, n_patterns_cleaned:]
sns.clustermap(corr, cmap='bwr',center=0, vmin=-0.7,vmax=0.7,annot=True)
plt.suptitle('Continuous variables, pearson correlation')

corr = patterns_cleaned_sub.merge(clinical_meta_cell.loc[:,cont_cats], left_index=True, right_index=True).corr('spearman').iloc[:n_patterns_cleaned, n_patterns_cleaned:]
sns.clustermap(corr, cmap='bwr',center=0, vmin=-0.7,vmax=0.7, annot=True)
plt.suptitle('Continuous variables, spearman correlation')


In [ ]:

sns.clustermap(patterns_cleaned_sub.merge(clinical_meta_cell_sub.loc[:,['Decreased Gestational Age', 'Decreased Postmenstrual Age at Death']], left_index=True, right_index=True).corr().iloc[:n_patterns_cleaned, n_patterns_cleaned:], cmap='bwr',center=0, vmin=-0.7,vmax=0.7,annot=True)
plt.suptitle('Continuous variables, pearson correlation')

sns.clustermap(patterns_cleaned_sub.merge(clinical_meta_cell_sub.loc[:,['Decreased Gestational Age', 'Decreased Postmenstrual Age at Death']], left_index=True, right_index=True).corr('spearman').iloc[:n_patterns_cleaned, n_patterns_cleaned:], cmap='bwr',center=0, vmin=-0.7,vmax=0.7,annot=True)
plt.suptitle('Continuous variables, spearman correlation')

In [ ]:
for cat in cats:
    
    merged = patterns_cleaned_sub.merge(pd.get_dummies(clinical_meta_cell.loc[:,cat]), left_index=True, right_index=True)
    
    sns.clustermap(merged.corr().iloc[:n_patterns_cleaned, n_patterns_cleaned:], cmap='bwr',center=0, vmin=-0.7,vmax=0.7,annot=True)
    plt.suptitle(cat + ', pearson corr', y=1, fontsize=12)

    sns.clustermap(merged.corr('spearman').iloc[:n_patterns_cleaned, n_patterns_cleaned:], cmap='bwr',center=0, vmin=-0.7,vmax=0.7,annot=True)
    plt.suptitle(cat + ', spearman corr', y=1, fontsize=12)